# Лабораторная работа №13. Распознавание рукописных цифр

[Задание](https://colab.research.google.com/drive/1qhqzI_rRUN3tP9gr_pd7_uNgtCiLuZ7u)

Обучаем десять логистических классификаторов «один против всех» на NumPy.
Каждая картинка — 400 чисел (20×20 пикселей). Обучение: 80%, тест: 20%.
Сигмоида, стоимость, регуляризация и градиентный спуск реализованы вручную.

**Источник данных:** файлы преподавателя не приложены к доступному Colab.
Использован открытый учебный набор `ex3data1.mat` из
[репозитория упражнений](https://github.com/dibgerge/ml-coursera-python-assignments/tree/master/Exercise3/Data),
преобразованный в `assets/lab13/data3_X.txt` и `data3_Y.txt`.
Размеры совпадают с заданием: 5000×400 и 5000×1; метка 10 обозначает ноль.
Тождественность файлам преподавателя не подтверждена. Их можно подставить по тем же путям.

Запуск: ядро Python (local models), ячейки сверху вниз. Нужны numpy и matplotlib.
API-ключи и интернет при повторном запуске не требуются.

## 1. Загрузка и подготовка

Одна общая перестановка индексов сохраняет соответствие картинки и метки.
Среднее и стандартное отклонение вычисляются только по обучающей части,
чтобы тестовые данные не участвовали в подготовке модели.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from io import BytesIO
from IPython.display import display, Image as NotebookImage

def show_figure(fig):
    # Явный PNG-вывод одинаково открывается в Jupyter и PyCharm.
    buffer = BytesIO()
    fig.savefig(buffer, format='png', dpi=120, bbox_inches='tight')
    display(NotebookImage(data=buffer.getvalue(), format='png'))

ROOT = Path.cwd()
if not (ROOT / 'assets').exists():
    ROOT = ROOT / 'Пересдача'
# Читаем таблицу пикселей: строка — картинка, столбец — один из 400 пикселей.
X = np.loadtxt(ROOT / 'assets/lab13/data3_X.txt')
# Читаем правильные классы; reshape(-1) превращает столбец в одномерный массив.
y = np.loadtxt(ROOT / 'assets/lab13/data3_Y.txt', dtype=int).reshape(-1)
y = np.where(y == 10, 0, y)  # В исходном наборе ноль записан числом 10.
assert X.shape == (5000, 400) and y.shape == (5000,)
assert np.isfinite(X).all() and set(y) == set(range(10))

# Фиксируем генератор случайных чисел для повторяемого разделения.
rng = np.random.default_rng(42)
# Перемешиваем номера строк, а затем используем их одновременно для X и y.
indices = rng.permutation(len(y))
# Первые 4000 индексов идут в обучение, оставшиеся 1000 — только в тест.
train_indices, test_indices = indices[:4000], indices[4000:]
X_train, X_test = X[train_indices], X[test_indices]
y_train, y_test = y[train_indices], y[test_indices]
# Среднее значение каждого пикселя вычисляем только на обучающей выборке.
mean = X_train.mean(axis=0)
# Стандартное отклонение задаёт масштаб каждого признака.
scale = X_train.std(axis=0)
scale[scale < 1e-8] = 1  # Постоянные пиксели не должны вызывать деление на ноль.
def prepare(images):
    # Приводим признаки к сопоставимым масштабам для градиентного спуска.
    normalized = (images - mean) / scale
    # Добавляем столбец единиц: его вес theta[0] — свободный коэффициент.
    return np.column_stack([np.ones(len(images)), normalized])
A_train, A_test = prepare(X_train), prepare(X_test)
assert not set(train_indices) & set(test_indices)
print('Исходные размеры:', X.shape, y.shape)
print('Обучение:', A_train.shape, 'Тест:', A_test.shape)
print('Классы в обучении:', np.bincount(y_train))

## 2. Математика классификатора

Для каждого класса k целевой ответ равен 1 для цифры k и 0 для остальных.
`z = X @ theta`, `p = sigmoid(z)`. Обучаем десять столбцов theta одновременно.

Стоимость каждого класса: средняя бинарная кросс-энтропия плюс
`λ/(2m) * sum(theta[1:]²)`. Свободный коэффициент theta[0] не штрафуется.
Градиент: `X.T @ (p - Y) / m + λ/m * theta` (кроме строки 0).
Обновление: `theta = theta - alpha * gradient`.
Для численной устойчивости стоимость считаем через `logaddexp`,
а сигмоиду — отдельно для положительных и отрицательных аргументов.

In [ ]:
# Сигмоида превращает линейную оценку z в число от 0 до 1.
def sigmoid(z):
    result = np.empty_like(z, dtype=float)
    # Разделяем аргументы по знаку, чтобы exp не переполнялась на больших числах.
    positive = z >= 0
    result[positive] = 1 / (1 + np.exp(-z[positive]))
    exponential = np.exp(z[~positive])
    result[~positive] = exponential / (1 + exponential)
    return result

def cost_and_gradient(A, targets, theta, regularization):
    # m — количество обучающих примеров, а не количество пикселей.
    m = len(A)
    # Матричное умножение: для каждой картинки считаем десять взвешенных сумм.
    z = A @ theta
    # Получаем оценки десяти независимых бинарных классификаторов.
    probabilities = sigmoid(z)
    # Отдельная копия нужна, чтобы исключить bias из штрафа, не меняя theta.
    penalty = theta.copy()
    penalty[0] = 0  # Свободный коэффициент не регуляризуем.
    # Устойчивая запись бинарной кросс-энтропии: log(1+exp(z)) - y*z.
    costs = (np.logaddexp(0, z) - targets * z).mean(axis=0)
    # Добавляем L2-штраф: большие веса увеличивают функцию стоимости.
    costs += regularization * (penalty ** 2).sum(axis=0) / (2 * m)
    # Производная стоимости по всем коэффициентам сразу: форма 401×10.
    gradient = A.T @ (probabilities - targets) / m
    # Добавляем производную L2-штрафа; нулевая строка penalty уже обнулена.
    gradient += regularization * penalty / m
    return costs, gradient

# Проверяем формулу градиента центральной конечной разностью на маленьком примере.
small_A = np.column_stack([np.ones(8), rng.normal(size=(8, 4))])
small_Y = rng.integers(0, 2, size=(8, 2))
small_theta = rng.normal(scale=0.1, size=(5, 2))
_, analytical = cost_and_gradient(small_A, small_Y, small_theta, 1.0)
numerical = np.zeros_like(small_theta)
epsilon = 1e-5
for row, col in np.ndindex(small_theta.shape):
    plus, minus = small_theta.copy(), small_theta.copy()
    plus[row, col] += epsilon
    minus[row, col] -= epsilon
    # Численно оцениваем производную по малому изменению одного коэффициента.
    numerical[row, col] = (
        cost_and_gradient(small_A, small_Y, plus, 1.0)[0].sum()
        - cost_and_gradient(small_A, small_Y, minus, 1.0)[0].sum()
    ) / (2 * epsilon)
assert np.allclose(analytical, numerical, atol=1e-7)
print('Проверка градиента пройдена; максимальная ошибка:', abs(analytical-numerical).max())

## 3. Обучение десяти моделей градиентным спуском

In [ ]:
# Матрица 4000×10: в строке единица только у правильной цифры, остальные нули.
targets = (y_train[:, None] == np.arange(10)[None, :]).astype(float)
# Начинаем с нулевых весов; каждый из десяти столбцов отвечает за свою цифру.
theta = np.zeros((A_train.shape[1], 10))
# Шаг обучения, сила L2-регуляризации и число обновлений весов.
alpha, regularization, iterations = 0.15, 1.0, 800
history = []
# Все десять классификаторов обучаются одновременно в одном цикле.
for step in range(iterations + 1):
    costs, gradient = cost_and_gradient(A_train, targets, theta, regularization)
    if step % 50 == 0:
        history.append((step, float(costs.mean())))
    if step < iterations:
        # Именно здесь происходит обучение: движемся против градиента стоимости.
        theta -= alpha * gradient
assert np.isfinite(theta).all()
assert history[-1][1] < history[0][1]
print('theta:', theta.shape, '— 401 коэффициент для каждой из 10 цифр')
print('Средняя стоимость: %.4f → %.4f' % (history[0][1], history[-1][1]))
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(*np.array(history).T)
ax.set(xlabel='Итерация', ylabel='Средняя стоимость', title='Обучение: уменьшение ошибки')
ax.grid(alpha=0.25)
fig.tight_layout()
show_figure(fig)
plt.close(fig)

## 4. Предсказания и качество на отложенном тесте

Каждый классификатор даёт оценку своего класса. `argmax` выбирает класс
с наибольшей оценкой. Десять независимых сигмоид не обязаны суммироваться в 1.
Тест не используется для обновления theta.

In [ ]:
def predict(images):
    # Используем обученные веса; при предсказании theta уже не обновляется.
    scores = sigmoid(prepare(images) @ theta)
    # Выбираем номер столбца с наибольшей оценкой: это предсказанная цифра.
    return np.argmax(scores, axis=1), scores

train_predictions, _ = predict(X_train)
test_predictions, test_scores = predict(X_test)
# Доля правильных ответов на знакомых модели обучающих примерах.
train_accuracy = np.mean(train_predictions == y_train)
# Доля правильных ответов на отложенных примерах, не участвовавших в обучении.
test_accuracy = np.mean(test_predictions == y_test)
print(f'Точность на обучении: {train_accuracy:.2%}')
print(f'Точность на тесте: {test_accuracy:.2%} ({np.sum(test_predictions == y_test)}/1000)')
confusion = np.zeros((10, 10), dtype=int)
# Строка матрицы — настоящий класс, столбец — предсказанный; на диагонали верные ответы.
np.add.at(confusion, (y_test, test_predictions), 1)
fig, ax = plt.subplots(figsize=(6, 5))
image = ax.imshow(confusion, cmap='Blues')
for row, col in np.ndindex(confusion.shape):
    ax.text(col, row, confusion[row, col], ha='center', va='center', fontsize=8,
            color='white' if confusion[row, col] > confusion.max()/2 else 'black')
ax.set(xticks=range(10), yticks=range(10), xlabel='Предсказанная цифра',
       ylabel='Настоящая цифра', title='Матрица ошибок на тесте')
fig.colorbar(image, ax=ax)
fig.tight_layout()
show_figure(fig)
plt.close(fig)

## 5. Распознавание конкретной картинки

In [ ]:
# Выбираем воспроизводимый случайный пример из теста.
i = int(np.random.default_rng(7).integers(len(y_test)))
digit = int(test_predictions[i])
print(f'Картинка №{i}: распознано {digit}, правильная цифра {y_test[i]}')
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
# В исходном MATLAB-наборе пиксели развёрнуты по столбцам.
axes[0].imshow(X_test[i].reshape(20, 20, order='F'), cmap='gray')
axes[0].set_title(f'Ответ модели: {digit}; метка: {y_test[i]}')
axes[0].axis('off')
axes[1].bar(range(10), test_scores[i])
axes[1].set(xticks=range(10), xlabel='Класс', ylabel='Оценка сигмоиды', ylim=(0, 1))
fig.tight_layout()
show_figure(fig)
plt.close(fig)

fig, axes = plt.subplots(3, 6, figsize=(9, 5))
for ax, index in zip(axes.flat, range(18)):
    ax.imshow(X_test[index].reshape(20, 20, order='F'), cmap='gray')
    correct = test_predictions[index] == y_test[index]
    ax.set_title(f'{y_test[index]} → {test_predictions[index]}', color='green' if correct else 'red')
    ax.axis('off')
fig.suptitle('Тест: настоящая цифра → предсказание')
fig.tight_layout()
show_figure(fig)
plt.close(fig)

## Что показать на защите

1. Загрузка: X — строки-картинки, y — правильные ответы; 10 заменяется на 0.
2. Одинаковые индексы перемешивают X и y; 4000 объектов обучают, 1000 проверяют.
3. `targets` — десять бинарных задач. Для класса 3 все тройки имеют ответ 1, остальные 0.
4. `cost_and_gradient` вычисляет стоимость и производную. Именно `theta -= alpha * gradient` обучает модель.
5. `predict` выбирает максимум из десяти оценок. В предсказании правильная метка не участвует.
6. Качество читаем из вывода тестовой ячейки. Ошибки показывают ограниченность линейной границы решения.

**Параметры:** alpha — шаг обучения, iterations — число обновлений,
regularization — штраф большим весам. Это обычный классификатор с 4010 обучаемыми
коэффициентами; веса получены на этих данных градиентным спуском.

**Ограничения:** один фиксированный split, подбор параметров по тесту не проводится;
на других изображениях потребуются одинаковые размер, ориентация и подготовка пикселей.
Сравнение train/test и матрица ошибок приведены выше с реальными результатами запуска.

## Вопросы преподавателя: от основных к уточняющим

**Что обучаем?** Десять бинарных логистических регрессий. Классификатор цифры 3
учится отделять тройки от всех остальных. Классификатор цифры 7 — семёрки от остальных.

**Что подаём на вход?** 400 чисел яркости пикселей. После нормировки и добавления
единицы получаем 401 признак. Модель не получает строку «это тройка».

**Что такое theta?** Обучаемые коэффициенты. Размер 401×10: 400 весов пикселей
и один свободный коэффициент для каждой цифры. Всего 4010 чисел.

**Где обучение?** `theta -= alpha * gradient`. До этой строки вычисляются оценки,
стоимость и её производная. После обновления веса немного лучше соответствуют данным.

**Зачем y?** Только для расчёта ошибки при обучении и проверки качества.
В `predict(images)` правильная метка не передаётся.

**Как появляется предсказание?** Нормируем 400 пикселей, добавляем единицу,
умножаем на theta, применяем сигмоиду к десяти оценкам и выбираем `argmax`.

**Что делает сигмоида?** Превращает любую линейную оценку в число от 0 до 1.
При z=0 результат 0.5; при z≈2.2 — около 0.9; при z≈−2.2 — около 0.1.
Это оценка принадлежности классу, а не гарантия правильности.

**Почему не порог 0.5?** Порог подходит для одной бинарной задачи. Здесь нужно
выдать одну из десяти цифр: максимум выбирается даже если все оценки ниже 0.5.

**Сумма десяти оценок равна 1?** Нет: классификаторы независимые. Это не softmax.
Две цифры могут одновременно получить высокие оценки; выбирается наибольшая.

**Зачем перемешивание?** Чтобы исходный порядок классов не определял состав train/test.
Если перемешать X и y независимо, картинки потеряют соответствие правильным меткам.

**Зачем тест?** Проверить способность распознавать примеры, на которых веса не обучались.
Нельзя обновлять theta по тестовым меткам или выбирать параметры по итоговой тестовой оценке.

**Зачем нормировка?** Выравнивает масштабы признаков и улучшает условия градиентного спуска.
На тест применяются среднее и масштаб обучения. Вычисление их на всём наборе — утечка данных.

**Зачем столбец единиц?** Он даёт свободный коэффициент b в формуле z=w·x+b.
Без него граница решения была бы дополнительно ограничена.

**Зачем регуляризация?** L2-штраф ограничивает большие веса и помогает против
переобучения. Слишком большой штраф приводит к недообучению. Bias не штрафуется.

**Зачем функция стоимости, если есть accuracy?** Accuracy считает дискретные ошибки,
а кросс-энтропия гладкая и даёт производную для обучения. Уверенный неверный ответ
штрафуется сильнее, чем неуверенный.

**Почему logaddexp?** Это численно устойчивая запись log(1+exp(z)); при больших z
прямой exp(z) может переполниться. Формула стоимости остаётся той же.

**Что делает alpha?** Определяет длину шага изменения весов. Слишком большой шаг
может мешать сходимости, слишком маленький замедляет обучение.

**Что проверяет численный градиент?** Сравнивает аналитическую производную с
разностью стоимостей при theta+epsilon и theta−epsilon. Это проверка формулы,
а не метод обучения на всём наборе.

**Почему нулевые веса допустимы?** Это независимые линейные классификаторы с разными
целевыми столбцами. У них разные градиенты. Проблема одинаковой инициализации скрытых
нейронов многослойной сети здесь не возникает.

**Что за reshape(order='F')?** Возвращает вектор пикселей в картинку 20×20 с учётом
порядка столбцов MATLAB. Это используется для отображения. Порядок признаков при
обучении и предсказании должен совпадать.

**Можно новую фотографию цифры?** Можно после подготовки к тому же виду: размер
20×20, яркость, фон, ориентация, размещение цифры и порядок пикселей. Просто подать
цветное фото произвольного размера в predict нельзя.

**Почему это регрессия, если выбираем цифру?** Логистическая регрессия моделирует
вероятность бинарного класса через сигмоиду; используется она для классификации.

**Это нейросеть?** По заданию это one-vs-all логистическая регрессия. Её можно
интерпретировать как десять логистических выходов без скрытых слоёв.

**Откуда 89.3%?** В сохранённом запуске верно распознано 893 из 1000 тестовых
картинок. На обучении — 93.85%. Это результат конкретных данных, разделения и параметров.
Он не означает такую же точность на любых фотографиях.

**Где лежат данные?** `assets/lab13/data3_X.txt` и `data3_Y.txt`.
Источник открытого учебного набора указан в начале. Совпадение с файлами преподавателя
не подтверждено; при их замене нужно заново выполнить ноутбук и оценить качество.
